# Speech generator

Sonify events with spoken phrases, using the `Speech` generator. Each source is spoken as one of a list of phrases, chosen by the `pitch` mapping. The phrases are rendered with text-to-speech up front (and cached), then played by the sampler machinery, so the usual generator parameters apply.

Needs the `kokoro` text-to-speech engine (`pip install strauss[speech]`), which the next cell installs on Colab.

In [ ]:
# on Colab, install the branch under test with kokoro (elsewhere, use your own install)
import sys
if 'google.colab' in sys.modules:
    !apt-get -qq -y install espeak-ng > /dev/null
    %pip --quiet install "strauss[speech] @ git+https://github.com/james-trayford/strauss.git@note-aliases"

In [ ]:
from strauss.sonification import Sonification
from strauss.sources import Events
from strauss.score import Score
from strauss.generator import Speech
from strauss.tts_caption import get_ttsMode
import numpy as np

if get_ttsMode() == 'None':
    raise RuntimeError("No text-to-speech engine found; install one with "
                       "'pip install strauss[speech]' to run this example.")

Some ordinal data (e.g. a weight class), with a time and a stereo panning for each event:

In [ ]:
odata = np.array([0., 2., 1., 0., 2., 1.])
time = np.linspace(0, 0.85, odata.size)
pan = np.array([0, 0.8, 0.1, 0.2, 0.9, 0.3])

The phrases to say, low to high in the `pitch` mapping - named in the score in place of notes, so sources are binned into them:

In [ ]:
phrases = ['low value', 'medium value', 'high value']
score = Score([phrases], '0m 10s')

Map the data, and build the `Speech` generator from the same phrases. `info()` lists the phrases it has rendered:

In [ ]:
data = {'pitch': odata,
        'pan': pan,
        'time': time}

sources = Events(data.keys())
sources.fromdict(data)
sources.apply_mapping_functions()

generator = Speech(phrases)
generator.info()

Render, and see which phrase each event speaks:

In [ ]:
soni = Sonification(score, sources, generator, 'stereo')
soni.render()
soni.event_table()

Listen:

In [ ]:
soni.notebook_display()

To write the audio out:

In [ ]:
# soni.save('./Speech.wav')